# 03 — Analyse visuelle : contrôle qualité, SAM 3.1, Muse Spark
Nécessite `MODEL_API_KEY`.

In [ ]:
# --- Bootstrap (Kaggle or local) -------------------------------------------------
# Kaggle: Settings > Internet = ON ; Add-ons > Secrets > add "MODEL_API_KEY" (and "GITHUB_TOKEN" if the repo is private)
import os, subprocess, sys
REPO = "code-htech07/sina-meta"
BRANCH = "ccr-c8fab4b3-0jmvi1"  # change to "main" once merged
if os.path.exists("engine"):            # already inside the repo (local run)
    pass
else:
    if not os.path.exists("sina-meta"):
        url = f"https://github.com/{REPO}.git"
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = f"https://{tok}@github.com/{REPO}.git"
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "sina-meta"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # silent: never print a tokenised URL
    os.chdir("sina-meta")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
from engine.config import describe_secret_status
from engine.logging_utils import setup_logging
setup_logging("WARNING")
print(describe_secret_status())  # shows only presence/length of the key, never its value

In [ ]:
from pathlib import Path
from engine.preprocessing.quality import assess_quality
samples = sorted(Path('data/samples/cocoaswolset').glob('*.jpg'))
for p in samples: q = assess_quality(p.stem, p); print(p.name, q.usable, q.issues, q.laplacian_variance)

In [ ]:
from engine.acquisition.images import load_rgb
from engine.meta_ai.sam import SamSegmenter
img = load_rgb('data/samples/cocoaswolset/cocoa_pod_Diseased.jpg')
seg = SamSegmenter().segment(img, 'nb03')
print(seg.status, seg.error, [(r.prompt, r.mask_area_fraction) for r in seg.regions])
from IPython.display import Image as I, display
if seg.regions: display(I(seg.regions[0].overlay_path))

In [ ]:
from engine.meta_ai.muse_spark import MuseSparkAnalyzer
from engine.preprocessing.quality import to_data_url
res = MuseSparkAnalyzer().analyze([to_data_url(img)], {'experiment': 'nb03_image_only'})
print(res.output.model_dump_json(indent=1)); print(res.usage)